# Multi-Agent Orchestration — Workflow Prototype

Interactive notebook for prototyping and testing multi-agent workflows.

**Kernel:** Use the project `.venv` (or parent repo venv once consolidated). The setup cell adds the project root to `sys.path` so `src` imports work from `notebooks/`.

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

In [2]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env", override=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {OUTPUT_DIR}")

Project root: /Users/vinayak/projects/agentic-dev-and-mcp/multi-agent-orchestration
Output dir:   /Users/vinayak/projects/agentic-dev-and-mcp/multi-agent-orchestration/output


In [19]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict 
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content


load_dotenv(override=True)

True

In [6]:
def _mask(value: str | None) -> str:
    if not value:
        return "(missing)"
    return value[:8] + "..." if len(value) > 8 else "(set)"

for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY"]:
    print(f"{key}: {_mask(os.getenv(key))}")

OPENAI_API_KEY: sk-proj-...
ANTHROPIC_API_KEY: sk-ant-a...
GOOGLE_API_KEY: AIzaSyAb...


## Testing Email 

In [10]:
def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("vinayak.kayaniv91@gmail.com")  # Change to your verified sender
    to_email = To("vinayak.kayaniv91@gmail.com")  # Change to your recipient
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()

202


## Step -1: Agent workflow

In [11]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

instructions4 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write warm, empathetic and friendly cold emails that are likely to get a response."

In [49]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model="gpt-4o-mini")
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model="gpt-4o-mini")
sales_agent3 = Agent(name="Busy Sales Agent", instructions=instructions3, model="gpt-4o-mini")
sales_agent4 = Agent(name="Friendly Sales Agent", instructions=instructions4, model="gpt-4o-mini")




In [53]:
sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x115b9bec0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [50]:
result = Runner.run_streamed(sales_agent2, input="Send a cold email to John Doe at Acme Inc.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Let’s Make SOC2 Compliance Less Painful Than a Root Canal! 🦷

Hi John,

I hope your day is going smoother than a cat on a Roomba!

I know you’re busy juggling reports and probably dodging audits like it’s a game of dodgeball, which is why I wanted to reach out. At ComplAI, we’re all about turning the SOC2 compliance circus into a walk in the park—kind of like a stroll with your favorite iced coffee, but without any of the spills.

Our AI-powered tool works harder than a caffeine-fueled squirrel on a mission, ensuring that you’re not just ready for audits but practically rolling out the red carpet for them. We handle everything from documentation to risk assessments so you can stop losing sleep over compliance and focus on what really matters: like figuring out where that elusive last donut in the break room went!

If you’re interested in turning compliance chaos into calm, let’s chat! I promise it’ll be more fun than a team-building retreat (and way more productive).

Looking 

In [21]:
import asyncio

In [35]:
from agents import tool


@function_tool
def send_email(email: str):
    """Send out an email with the given body to all sales prospects"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("vinayak.kayaniv91@gmail.com")  # Change to your verified sender
    to_email = To("vinayak.kayaniv91@gmail.com")  # Change to your recipient
    content = Content("text/plain", email)
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return response.status_code
    
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'email': {'title': 'Email', 'type': 'string'}}, 'required': ['email'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1153abb00>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [31]:
sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1153aa980>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [56]:
description = "Write a cold sales email"
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)
tool4 = sales_agent4.as_tool(tool_name="sales_agent4", tool_description=description)
tools = [tool1, tool2, tool3, tool4, send_email]



In [58]:
sales_agent1

Agent(name='Professional Sales Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

In [59]:
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x115c19300>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [ ]:
sales_agent1({
  "input": "Dear CEO,\n\nI hope this message finds you well. At ComplAI, we specialize in streamlining operations with AI-driven solutions that empower businesses to operate more efficiently and effectively. \n\nI would love the opportunity to discuss how our innovative technologies can help your company reduce costs and enhance productivity. Are you available for a brief chat this week?\n\nLooking forward to your response.\n\nBest regards,\n\n[Your Name]\n[Your Position]\nComplAI\n[Your Contact Information]"
})

In [48]:
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x115b98400>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [63]:
sales_manager_instructions = """You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Request Drafts: Simply ask the 4 sales_agent tools to create 4 different email drafts. Do not give them an example email draft, JUST ASK FOR THEM TO WRITE ONE with a simple prompt. 
Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not give them a draft as input prompt PLEASE!
- You must send ONE email using the send_email tool — never more than one."""

sales_manager_instructions

'You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.\n\nFollow these steps carefully:\n1. Request Drafts: Simply ask the 4 sales_agent tools to create 4 different email drafts. Do not give them an example email draft, JUST ASK FOR THEM TO WRITE ONE with a simple prompt. \nDo not proceed until all three drafts are ready.\n\n2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.\n\n3. Use the send_email tool to send the best email (and only the best email) to the user.\n\nCrucial Rules:\n- You must use the sales agent tools to generate the drafts — do not give them a draft as input prompt PLEASE!\n- You must send ONE email using the send_email tool — never more than one.'

In [ ]:
sales_manager = Agent(name="Sales Manager", instructions=sales_manager_instructions, model="gpt-4o-mini", tools=tools)

with trace("Sales Manager"):
    result = await Runner.run(sales_manager, "Send a cold sales email addressed to 'Dear CEO'")

print(result.final_output)


The cold sales email has been successfully sent. If you need any more assistance or if there's anything else you'd like to do, just let me know!


In [69]:
subject_line_instructions = """You are a subject line agent. 
You are responsible for generating an email subject line which can catch attention of the reader. 
You are given the email body as an input. Your job is to only generate the subject line and nothing else."""

html_converter_instructions = """You are a html email converter agent. You are given a text email, your job is to convert the email into html format.
There might be some formatting issues in the text email, you need to fix them and convert the email into html format."""

subject_line_agent = Agent(name="Subject Line Agent", instructions=subject_line_instructions, model="gpt-4o-mini")
html_converter_agent = Agent(name="HTML Converter Agent", instructions= html_converter_instructions, model="gpt-4o-mini")


In [76]:
generate_subject_line = subject_line_agent.as_tool(tool_name="generate_subject_line", tool_description="Generate a appropriate subject line for the email which can catch attention of the reader")
convert_to_html = html_converter_agent.as_tool(tool_name="convert_to_html", tool_description="Convert the email body into html format")


In [67]:
@function_tool
def send_html_email(subject_line: str, html_body: str):
    """Send out an html formatted email with the given subject line and body to the user"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("vinayak.kayaniv91@gmail.com")  # Change to your verified sender
    to_email = To("vinayak.kayaniv91@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject_line, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return response.status_code

send_html_email

FunctionTool(name='send_html_email', description='Send out an html formatted email with the given subject line and body to the user', params_json_schema={'properties': {'subject_line': {'title': 'Subject Line', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject_line', 'html_body'], 'title': 'send_html_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x115b99260>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [77]:
tools = [generate_subject_line, convert_to_html, send_html_email]

In [78]:
emailer_agent_instructions = """ You are an emailer agent. You are responsible for creating a subject line by calling the generate_subject_line tool. 
Then converting the email body into html format by calling the convert_to_html tool. 
Finally you need to call the send email tool to send the html formatted email with the subject line."""

emailer_agent = Agent(
    name="Emailer Agent",
    instructions=emailer_agent_instructions,
    model="gpt-4o-mini",
    tools=tools,
    handoff_description="Convert an email to HTML and send it"
)

In [79]:
handoffs = [emailer_agent]


In [80]:
tools = [tool1, tool2, tool3, tool4]

In [81]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

In [82]:
sales_manager_instructions

sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    model="gpt-4o-mini",
    tools=tools,
    handoffs=handoffs,
)

In [83]:
sales_manager

Agent(name='Sales Manager', handoff_description=None, tools=[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x115c19300>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x115c1a340>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=No

In [84]:
message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Sales Manager"):
    result = await Runner.run(sales_manager, message)

print(result.final_output)


The email has been successfully sent to the CEO with the subject line "Boost Your SOC 2 Compliance Effortlessly with ComplAI!" 

If you need anything else or wish to make further adjustments, feel free to ask!


## Smoke test

Quick check that API access works before building a multi-agent workflow.

In [ ]:
from openai import OpenAI

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "Reply with one short sentence confirming you are ready to help prototype a multi-agent workflow.",
        }
    ],
    max_tokens=60,
)

response.choices[0].message.content